# Experiment 16 — Gated DeltaNet

Third notebook in the subquadratic-attention track (see experiment 14's intro for the
full framing; same `K`-pair multi-query associative recall test used across 14-17).
Experiments 14 (Mamba) and 15 (RWKV) both write into their recurrent state by **adding**:
each new token's contribution gets summed (with a decay) into the running state,
channel by channel. That has a structural weakness for exact recall — nothing ever
*removes* an old, no-longer-relevant contribution, so writes pile up and interfere with
each other, especially when the state has to hold several arbitrary key→value bindings
at once, which is exactly what the MQAR task demands.

**Gated DeltaNet** (Yang, Wang, Shen, Panda & Kim, 2024, "Gated Delta Networks: Improving
Mamba2 with Delta Rule") fixes this by giving the state real **matrix** structure — not
one number per channel, but a full associative-memory matrix `S` (key-space ×
value-space) per head — and replacing additive writes with the **delta rule** (Widrow-Hoff
/ least-mean-squares, and specifically its use as a linear "fast weight" memory in Schlag
et al. 2021 and Yang et al. 2024's parallelized DeltaNet): at every step, first *read* what
the memory currently associates with this key, compare it to the value that should
actually be there, and write only the **error**. Gated DeltaNet adds one more piece
(borrowed from Mamba2): a learned scalar decay gate that lets old associations fade when
they're no longer useful — "gated" delta rule.

## The delta rule as an error-correcting write

Plain (ungated) associative writes — what a Hopfield network or vanilla linear attention
does — look like `S_t = S_{t-1} + k_t v_t^T`: just keep adding outer products. If the same
key (or a similar one) shows up twice with different values, both writes coexist in `S`
and interfere — reading back with that key returns a blend of both, not either one
cleanly.

The delta rule instead asks: *what does the memory currently think the value for this key
is*, and only writes the difference:

```
v_pred_t = S_{t-1}^T k_t              # what the memory currently retrieves for this key
delta_t  = v_t - v_pred_t             # the error: what's actually missing
S_t      = alpha_t * S_{t-1} + beta_t * k_t @ delta_t^T
```

`beta_t` (a learned, data-dependent, sigmoid-bounded "write strength") controls how much
of the error actually gets written — this is literally one step of gradient descent on
the squared-error loss `||S^T k_t - v_t||^2`, per token. `alpha_t` is the Gated-DeltaNet
addition: a learned scalar decay gate (same role as Mamba's selective `Δ`) that lets
stale key→value bindings fade instead of persisting forever. Keys are L2-normalized
before use, a detail from the DeltaNet papers that keeps the write's magnitude
well-behaved regardless of the raw projection's scale.

Read-out at query time is a simple matrix-vector product against the query, `y_t = S_t
q_t` — no decay, no history walk, just "look up what's stored for this query right now."

Like the previous two notebooks, `q`/`k`/`v` here go through a short causal depthwise
conv1d before the recurrence (same reason as Mamba's conv / RWKV's token-shift: a token
needs to see the one just before it, so a value token's write can actually bind to the
key it followed — confirmed necessary in experiment 14, this notebook has it from the
start).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- same MQAR task as experiments 14/15, copied rather than imported (one-file-per-notebook rule) ---
K = 8
M = 4
SEP = 1
KEY0 = 2
VAL0 = 2 + K
VOCAB = 2 + 2 * K


def make_batch(batch_size, device="cpu"):
    value_assignment = torch.argsort(torch.rand(batch_size, K), dim=1)
    order = torch.argsort(torch.rand(batch_size, K), dim=1)
    key_ids = KEY0 + order
    val_ids = VAL0 + torch.gather(value_assignment, 1, order)
    context = torch.stack([key_ids, val_ids], dim=2).reshape(batch_size, 2 * K)
    sep = torch.full((batch_size, 1), SEP, dtype=torch.long)
    q_key_identity = torch.randint(0, K, (batch_size, M))
    query_tokens = KEY0 + q_key_identity
    query_labels = torch.gather(value_assignment, 1, q_key_identity)
    seq = torch.cat([context, sep, query_tokens], dim=1)
    return seq.to(device), query_labels.to(device)


seq, labels = make_batch(1)
print("one example sequence:", seq[0].tolist())
print("correct value-class for each query:", labels[0].tolist())


device: cuda
one example sequence: [6, 10, 7, 15, 8, 11, 4, 14, 5, 16, 2, 12, 9, 17, 3, 13, 1, 2, 6, 4, 3]
correct value-class for each query: [2, 0, 4, 3]


In [2]:
class GatedDeltaNetLayer(nn.Module):
    def __init__(self, d_model, n_heads=4, conv_k=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.dh = d_model // n_heads
        self.conv_k = conv_k
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        # short causal conv on q/k/v, same role as Mamba's conv / RWKV's token-shift
        self.q_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.k_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.v_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.beta_proj = nn.Linear(d_model, n_heads)     # write strength, scalar per head
        self.alpha_proj = nn.Linear(d_model, n_heads)    # decay gate, scalar per head
        self.out_proj = nn.Linear(d_model, d_model)

    def _causal_conv(self, conv, x):
        T = x.shape[1]
        x_pad = F.pad(x.transpose(1, 2), (self.conv_k - 1, 0))
        return F.silu(conv(x_pad).transpose(1, 2))

    def forward(self, x):
        B, T, D = x.shape
        H, dh = self.n_heads, self.dh
        q = self._causal_conv(self.q_conv, self.q_proj(x)).view(B, T, H, dh)
        k = F.normalize(self._causal_conv(self.k_conv, self.k_proj(x)).view(B, T, H, dh), dim=-1)
        v = self._causal_conv(self.v_conv, self.v_proj(x)).view(B, T, H, dh)
        beta = torch.sigmoid(self.beta_proj(x))    # (B,T,H)
        alpha = torch.sigmoid(self.alpha_proj(x))  # (B,T,H)

        S = x.new_zeros(B, H, dh, dh)   # per-head associative memory: key-space x value-space
        outs = []
        for t in range(T):
            kt, vt, qt = k[:, t], v[:, t], q[:, t]
            bt = beta[:, t].unsqueeze(-1).unsqueeze(-1)
            at = alpha[:, t].unsqueeze(-1).unsqueeze(-1)

            v_pred = torch.einsum("bhk,bhkv->bhv", kt, S)   # what's currently stored for this key
            delta = vt - v_pred                              # the error
            S = at * S + bt * torch.einsum("bhk,bhv->bhkv", kt, delta)   # gated delta-rule write
            y_t = torch.einsum("bhk,bhkv->bhv", qt, S)        # read-out for the query
            outs.append(y_t.reshape(B, D))
        y = torch.stack(outs, dim=1)
        return self.out_proj(y)


## Wiring it into a tiny sequence model

Same wrapper as experiment 14: two stacked layers, pre-norm residual, classifier head
reading every position, loss computed only at the `M` query positions. Same
hyperparameters too (`d_model=64`, 2 layers) so the comparison to Mamba/RWKV isn't
confounded by giving this one more capacity.

In [3]:
class TokenModel(nn.Module):
    def __init__(self, layer_factory, d_model, n_layers, vocab=VOCAB, n_classes=K):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList([layer_factory() for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, tokens):
        x = self.embed(tokens)
        for layer, norm in zip(self.layers, self.norms):
            x = x + layer(norm(x))
        return self.head(self.final_norm(x))


def train_and_eval(model, steps=1500, batch_size=64, lr=3e-3, n_queries=M, log_every=300):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        seq, labels = make_batch(batch_size, device)
        logits = model(seq)[:, -n_queries:, :]
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            acc = (logits.argmax(-1) == labels).float().mean().item()
            print(f"step {step:4d}  loss {loss.item():.4f}  train_acc {acc:.3f}")

    model.eval()
    with torch.no_grad():
        seq, labels = make_batch(2000, device)
        logits = model(seq)[:, -n_queries:, :]
        test_acc = (logits.argmax(-1) == labels).float().mean().item()
    print(f"\nFINAL TEST ACC: {test_acc:.4f}   (chance = {1/K:.4f})")
    return test_acc


torch.manual_seed(0)
gdn_model = TokenModel(lambda: GatedDeltaNetLayer(64, n_heads=4), d_model=64, n_layers=2)
n_params = sum(p.numel() for p in gdn_model.parameters())
print(f"params: {n_params:,}")
test_acc = train_and_eval(gdn_model)


params: 38,296


step    0  loss 2.2249  train_acc 0.137


step  300  loss 0.0014  train_acc 1.000


step  600  loss 0.0004  train_acc 1.000


step  900  loss 0.0002  train_acc 1.000


step 1200  loss 0.0001  train_acc 1.000


step 1499  loss 0.0001  train_acc 1.000

FINAL TEST ACC: 1.0000   (chance = 0.1250)


## What actually happened

**No plateau, no phase transition — the gated delta rule solved the task almost
immediately.** Train accuracy was already at 1.000 by step 300 (loss 0.0014) and stayed
there for the rest of training. Final held-out test accuracy: **1.0000** on 2,000 fresh
examples (chance = 0.1250), with 38,296 parameters — fewer than either experiment 14
(42,184) or 15 (110,984), and dramatically faster to converge than both.

**This is the real, expected result, not a coincidence.** Experiments 14 and 15 both
write into their state by *adding* a decayed contribution every step — there's no
mechanism to notice "this key already has something stored, and it's wrong" and correct
it, so the model has to discover, purely through gradient descent on the whole
recurrence, some indirect scheme where addition-based writes still end up separable at
read time. That took on the order of a thousand-plus steps in both cases. The delta
rule bakes the correction in structurally: `v_pred = S^T k`, `delta = v - v_pred` **is
already** "notice what's wrong", every single step, before any training happens at all.
Gradient descent then only has to learn good `k`/`v`/`q` projections and reasonable
`beta`/`alpha` gates — the error-correcting write itself doesn't need to be discovered.
This matches exactly what the Gated DeltaNet paper (and the DeltaNet/Based/Zoology line of
work it builds on) uses this exact task to demonstrate.

**Comparing to experiments 14/15 so far:** all three architectures reach the identical
1.0000 ceiling on this toy task — the difference is entirely in how many steps it takes to
get there (Gated DeltaNet: ~300, Mamba: ~1200, RWKV: ~2000), not in some models
succeeding and others failing. Whether that convergence-speed gap turns into an actual
accuracy gap once the task is made harder — more pairs than the state can comfortably
hold — is exactly what experiment 17's capacity test investigates.

**What this does and doesn't show:** same caveats as 14/15 — single seed, toy scale
(`K=8`, `T=21`), sequential Python recurrence rather than a fused kernel, and a single
`beta`/`alpha` scalar per head rather than the finer per-channel gating Kimi Linear (17)
introduces.